# Step 01_02_04 -- Metadata STRUCT Extraction & Replay-Level EDA

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA (Tukey-style)
**Dataset:** sc2egset
**Question:** What scalar fields are embedded in the metadata STRUCTs,
what are their value distributions, and what is the full column-level
profile (NULLs, constants, cardinality) across all raw tables?
**Invariants applied:** #6 (reproducibility — SQL inlined in artifact),
#7 (no magic numbers), #9 (step scope: univariate profiling only)
**Step scope:** query, profile, visualize
**Type:** Read-only — no DuckDB writes, no new tables, no schema changes

In [1]:
import json
from pathlib import Path

import duckdb
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from rts_predict.common.eda_census import profile_table
from rts_predict.common.notebook_utils import get_reports_dir, setup_notebook_logging
from rts_predict.games.sc2.config import DB_FILE

matplotlib.use("Agg")
logger = setup_notebook_logging()
logger.info("DB_FILE: %s", DB_FILE)

22:25:12 INFO notebook: DB_FILE: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/db.duckdb


In [2]:
con = duckdb.connect(str(DB_FILE), read_only=True)
print(f"Connected (read-only): {DB_FILE}")

Connected (read-only): /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/db.duckdb


### Database memory footprint

In [3]:
import os
db_size_bytes = os.path.getsize(str(DB_FILE))
db_size_mb = round(db_size_bytes / (1024 * 1024), 2)
print(f"DuckDB file size: {db_size_bytes:,} bytes ({db_size_mb} MB)")

DuckDB file size: 12,333,056 bytes (11.76 MB)


In [4]:
# Set up artifact directories
artifacts_dir = (
    get_reports_dir("sc2", "sc2egset")
    / "artifacts" / "01_exploration" / "02_eda"
)
plots_dir = artifacts_dir / "plots"
artifacts_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"Artifacts dir: {artifacts_dir}")
print(f"Plots dir: {plots_dir}")

Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda
Plots dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda/plots


## Section A: STRUCT Field Extraction

Flatten the four STRUCT columns (`details`, `header`, `initData`,
`metadata`) from `replays_meta_raw` into scalar fields.

In [5]:
STRUCT_FLAT_SQL = """\
SELECT
    details.gameSpeed AS game_speed,
    details.isBlizzardMap AS is_blizzard_map,
    details.timeUTC AS time_utc,
    header.elapsedGameLoops AS elapsed_game_loops,
    header."version" AS game_version_header,
    metadata.baseBuild AS base_build,
    metadata.dataBuild AS data_build,
    metadata.gameVersion AS game_version_meta,
    metadata.mapName AS map_name,
    initData.gameDescription.maxPlayers AS max_players,
    initData.gameDescription.gameSpeed AS game_speed_init,
    initData.gameDescription.isBlizzardMap AS is_blizzard_map_init,
    initData.gameDescription.mapSizeX AS map_size_x,
    initData.gameDescription.mapSizeY AS map_size_y,
    gameEventsErr,
    messageEventsErr,
    trackerEvtsErr,
    filename
FROM replays_meta_raw
"""
struct_flat = con.execute(STRUCT_FLAT_SQL).df()
print(f"struct_flat shape: {struct_flat.shape}")
print(struct_flat.head())

struct_flat shape: (22390, 18)
  game_speed  is_blizzard_map                      time_utc  \
0     Faster             True  2016-01-31T19:13:30.0627872Z   
1     Faster             True  2016-02-01T23:57:36.6854685Z   
2     Faster             True  2016-02-01T23:29:49.0225545Z   
3     Faster             True  2016-02-01T21:21:17.0189823Z   
4     Faster             True  2016-01-31T21:19:55.1928005Z   

   elapsed_game_loops game_version_header base_build data_build  \
0                8550         3.1.1.39948  Base39948      39948   
1                7501         3.1.1.39948  Base39948      39948   
2               12943         3.1.1.39948  Base39948      39948   
3               22182         3.1.1.39948  Base39948      39948   
4                7568         3.1.1.39948  Base39948      39948   

  game_version_meta          map_name  max_players game_speed_init  \
0       3.1.1.39948     Lerilak Crest            4          Faster   
1       3.1.1.39948  Central Protocol          

## Section B: Full NULL Census

All 25 `replay_players_raw` columns + `replays_meta_raw.filename`.

In [6]:
NULL_CENSUS_SQL = """\
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(filename) AS filename_null,
    COUNT(*) - COUNT(toon_id) AS toon_id_null,
    COUNT(*) - COUNT(nickname) AS nickname_null,
    COUNT(*) - COUNT(playerID) AS playerID_null,
    COUNT(*) - COUNT(userID) AS userID_null,
    COUNT(*) - COUNT(isInClan) AS isInClan_null,
    COUNT(*) - COUNT(clanTag) AS clanTag_null,
    COUNT(*) - COUNT(MMR) AS MMR_null,
    COUNT(*) - COUNT(race) AS race_null,
    COUNT(*) - COUNT(selectedRace) AS selectedRace_null,
    COUNT(*) - COUNT(handicap) AS handicap_null,
    COUNT(*) - COUNT(region) AS region_null,
    COUNT(*) - COUNT(realm) AS realm_null,
    COUNT(*) - COUNT(highestLeague) AS highestLeague_null,
    COUNT(*) - COUNT(result) AS result_null,
    COUNT(*) - COUNT(APM) AS APM_null,
    COUNT(*) - COUNT(SQ) AS SQ_null,
    COUNT(*) - COUNT(supplyCappedPercent) AS supplyCappedPercent_null,
    COUNT(*) - COUNT(startDir) AS startDir_null,
    COUNT(*) - COUNT(startLocX) AS startLocX_null,
    COUNT(*) - COUNT(startLocY) AS startLocY_null,
    COUNT(*) - COUNT(color_a) AS color_a_null,
    COUNT(*) - COUNT(color_b) AS color_b_null,
    COUNT(*) - COUNT(color_g) AS color_g_null,
    COUNT(*) - COUNT(color_r) AS color_r_null
FROM replay_players_raw
"""
null_raw = con.execute(NULL_CENSUS_SQL).df()
total_rows = int(null_raw["total_rows"].iloc[0])
print(f"Total rows in replay_players_raw: {total_rows}")

Total rows in replay_players_raw: 44817


In [7]:
# Reshape into tidy (column, null_count, null_pct) table
null_cols = [c for c in null_raw.columns if c.endswith("_null")]
null_records = []
for col in null_cols:
    col_name = col.replace("_null", "")
    null_count = int(null_raw[col].iloc[0])
    null_pct = round(100.0 * null_count / total_rows, 2)
    null_records.append(
        {"column": col_name, "null_count": null_count, "null_pct": null_pct}
    )

null_census_df = pd.DataFrame(null_records)
null_census_df = null_census_df.sort_values("null_count", ascending=False)
print("=== replay_players_raw NULL census ===")
print(null_census_df.to_string(index=False))

=== replay_players_raw NULL census ===
             column  null_count  null_pct
           filename           0       0.0
      highestLeague           0       0.0
            color_g           0       0.0
            color_b           0       0.0
            color_a           0       0.0
          startLocY           0       0.0
          startLocX           0       0.0
           startDir           0       0.0
supplyCappedPercent           0       0.0
                 SQ           0       0.0
                APM           0       0.0
             result           0       0.0
              realm           0       0.0
            toon_id           0       0.0
             region           0       0.0
           handicap           0       0.0
       selectedRace           0       0.0
               race           0       0.0
                MMR           0       0.0
            clanTag           0       0.0
           isInClan           0       0.0
             userID           0      

In [8]:
# Check replays_meta_raw.filename NULLs
META_FILENAME_NULL_SQL = """\
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(filename) AS filename_null,
    ROUND(100.0 * (COUNT(*) - COUNT(filename)) / COUNT(*), 2) AS filename_null_pct
FROM replays_meta_raw
"""
meta_fn_null = con.execute(META_FILENAME_NULL_SQL).df()
print("=== replays_meta_raw.filename NULL census ===")
print(meta_fn_null.to_string(index=False))

=== replays_meta_raw.filename NULL census ===
 total_rows  filename_null  filename_null_pct
      22390              0                0.0


### struct_flat NULL census

In [9]:
STRUCT_FLAT_NULL_CENSUS_SQL = """\
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(time_utc) AS time_utc_null,
    COUNT(*) - COUNT(elapsed_game_loops) AS elapsed_game_loops_null,
    COUNT(*) - COUNT(game_version_header) AS game_version_header_null,
    COUNT(*) - COUNT(base_build) AS base_build_null,
    COUNT(*) - COUNT(data_build) AS data_build_null,
    COUNT(*) - COUNT(game_version_meta) AS game_version_meta_null,
    COUNT(*) - COUNT(map_name) AS map_name_null,
    COUNT(*) - COUNT(max_players) AS max_players_null,
    COUNT(*) - COUNT(game_speed) AS game_speed_null,
    COUNT(*) - COUNT(game_speed_init) AS game_speed_init_null,
    COUNT(*) - COUNT(is_blizzard_map) AS is_blizzard_map_null,
    COUNT(*) - COUNT(is_blizzard_map_init) AS is_blizzard_map_init_null,
    COUNT(*) - COUNT(map_size_x) AS map_size_x_null,
    COUNT(*) - COUNT(map_size_y) AS map_size_y_null,
    COUNT(*) - COUNT(gameEventsErr) AS gameEventsErr_null,
    COUNT(*) - COUNT(messageEventsErr) AS messageEventsErr_null,
    COUNT(*) - COUNT(trackerEvtsErr) AS trackerEvtsErr_null
FROM struct_flat
"""
sf_null_raw = con.execute(STRUCT_FLAT_NULL_CENSUS_SQL).df()
sf_total_rows = int(sf_null_raw["total_rows"].iloc[0])
print(f"Total rows in struct_flat: {sf_total_rows}")

Total rows in struct_flat: 22390


In [10]:
# Reshape into tidy (column, null_count, null_pct) table
sf_null_cols = [c for c in sf_null_raw.columns if c.endswith("_null")]
sf_null_records = []
for col in sf_null_cols:
    col_name = col.replace("_null", "")
    null_count = int(sf_null_raw[col].iloc[0])
    null_pct = round(100.0 * null_count / sf_total_rows, 2)
    sf_null_records.append(
        {"column": col_name, "null_count": null_count, "null_pct": null_pct}
    )

sf_null_census_df = pd.DataFrame(sf_null_records)
sf_null_census_df = sf_null_census_df.sort_values("null_count", ascending=False)
print("=== struct_flat NULL census ===")
print(sf_null_census_df.to_string(index=False))

=== struct_flat NULL census ===
              column  null_count  null_pct
            time_utc           0       0.0
     game_speed_init           0       0.0
    messageEventsErr           0       0.0
       gameEventsErr           0       0.0
          map_size_y           0       0.0
          map_size_x           0       0.0
is_blizzard_map_init           0       0.0
     is_blizzard_map           0       0.0
          game_speed           0       0.0
  elapsed_game_loops           0       0.0
         max_players           0       0.0
            map_name           0       0.0
   game_version_meta           0       0.0
          data_build           0       0.0
          base_build           0       0.0
 game_version_header           0       0.0
      trackerEvtsErr           0       0.0


### struct_flat completeness note

In [11]:
_sf_all_zero_nulls = (sf_null_census_df["null_count"] == 0).all()
if _sf_all_zero_nulls:
    sf_completeness_note = "All 17 columns 0% NULL. No missingness co-occurrence."
    print("All 17 struct_flat columns have 0 NULLs. No missingness co-occurrence to analyze.")
else:
    # Build co-occurrence matrix for columns that have NULLs
    _null_cols_list = sf_null_census_df[sf_null_census_df["null_count"] > 0]["column"].tolist()
    _cooc_data = {}
    for col_a in _null_cols_list:
        _cooc_data[col_a] = {}
        for col_b in _null_cols_list:
            sql = f"""
            SELECT COUNT(*) AS both_null
            FROM struct_flat
            WHERE {col_a} IS NULL AND {col_b} IS NULL
            """
            both_null = con.execute(sql).fetchone()[0]
            _cooc_data[col_a][col_b] = both_null
    sf_completeness_note = {"type": "co_occurrence", "matrix": _cooc_data}
    _cooc_df = pd.DataFrame(_cooc_data)
    print("=== struct_flat NULL co-occurrence matrix ===")
    print(_cooc_df.to_string())

All 17 struct_flat columns have 0 NULLs. No missingness co-occurrence to analyze.


## Section C: Target Variable (`result`)

In [12]:
RESULT_DIST_SQL = """\
SELECT result, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM replay_players_raw
GROUP BY result
ORDER BY cnt DESC
"""
result_dist = con.execute(RESULT_DIST_SQL).df()
print("=== result value distribution ===")
print(result_dist.to_string(index=False))

=== result value distribution ===
   result   cnt   pct
     Loss 22409 50.00
      Win 22382 49.94
Undecided    24  0.05
      Tie     2  0.00


In [13]:
# Isolate results for non-2-player replays (known 11 from 01_02_02)
NON_2P_RESULT_SQL = """\
SELECT rp.result, COUNT(*) AS cnt
FROM replay_players_raw rp
JOIN replays_meta_raw rm ON rp.filename = rm.filename
WHERE rm.initData.gameDescription.maxPlayers != 2
   OR rm.filename IN (
       SELECT filename FROM replay_players_raw
       GROUP BY filename HAVING COUNT(*) != 2
   )
GROUP BY rp.result
ORDER BY cnt DESC
"""
non_2p_results = con.execute(NON_2P_RESULT_SQL).df()
print("=== result values for non-2-player replays ===")
print(non_2p_results.to_string(index=False))
print(
    "\nNote: OR-condition captures a superset -- any replay where "
    "either maxPlayers != 2 or actual player-row count differs from 2."
)

=== result values for non-2-player replays ===
result  cnt
  Loss  444
   Win  417

Note: OR-condition captures a superset -- any replay where either maxPlayers != 2 or actual player-row count differs from 2.


In [14]:
# Corrective query: Undecided/Tie replay context
# The OR-condition non-2-player query above returns only Win/Loss.
# This query uses LEFT JOIN so Undecided/Tie rows are included even if
# filename is absent from replays_meta_raw.
# Note: Joins on filename per sql-data.md rule 15 deviation -- unavoidable at this
# pipeline stage (replay_id not yet derived). Migrates to replay_id in 01_04.
UNDECIDED_TIE_CONTEXT_SQL = """\
SELECT
    rp.result,
    rp.filename,
    rm.initData.gameDescription.maxPlayers AS max_players,
    player_counts.player_row_count
FROM replay_players_raw rp
LEFT JOIN replays_meta_raw rm ON rp.filename = rm.filename
JOIN (
    SELECT filename, COUNT(*) AS player_row_count
    FROM replay_players_raw
    GROUP BY filename
) player_counts ON rp.filename = player_counts.filename
WHERE rp.result IN ('Undecided', 'Tie')
ORDER BY rp.result, rp.filename
"""
undecided_tie_context = con.execute(UNDECIDED_TIE_CONTEXT_SQL).df()
print(f"=== Undecided/Tie replay context ({len(undecided_tie_context)} rows) ===")
print(undecided_tie_context.to_string(index=False))

=== Undecided/Tie replay context (26 rows) ===
   result                                                                                                                                   filename  max_players  player_row_count
      Tie               2022_Dreamhack_SC2_Masters_Valencia/2022_Dreamhack_SC2_Masters_Valencia_data/cab0125665bd7ccf10270d10cf214463.SC2Replay.json            2                 2
      Tie               2022_Dreamhack_SC2_Masters_Valencia/2022_Dreamhack_SC2_Masters_Valencia_data/cab0125665bd7ccf10270d10cf214463.SC2Replay.json            2                 2
Undecided                                               2017_WESG_Barcelona/2017_WESG_Barcelona_data/d71ff1dbe2f741f150d9baeee55346b9.SC2Replay.json            2                 2
Undecided                                               2017_WESG_Barcelona/2017_WESG_Barcelona_data/d71ff1dbe2f741f150d9baeee55346b9.SC2Replay.json            2                 2
Undecided                                       2018_

## Section D: Categorical Field Profiles

Distinct value counts for categorical fields using loop-based cells.

In [15]:
# replay_players_raw categorical fields
RP_CAT_COLS = ["race", "selectedRace", "highestLeague", "region", "realm"]
rp_cat_results = {}

for col in RP_CAT_COLS:
    sql = f"""
    SELECT {col}, COUNT(*) AS cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM replay_players_raw
    GROUP BY {col}
    ORDER BY cnt DESC
    """
    df = con.execute(sql).df()
    rp_cat_results[col] = df
    print(f"\n=== {col} ===")
    print(df.to_string(index=False))


=== race ===
race   cnt   pct
Prot 16228 36.21
Zerg 15695 35.02
Terr 12891 28.76
BWTe     1  0.00
BWZe     1  0.00
BWPr     1  0.00

=== selectedRace ===
selectedRace   cnt   pct
        Prot 15948 35.58
        Zerg 15123 33.74
        Terr 12623 28.17
              1110  2.48
        Rand    10  0.02
        BWTe     1  0.00
        BWZe     1  0.00
        BWPr     1  0.00

=== highestLeague ===
highestLeague   cnt   pct
      Unknown 32338 72.16
       Master  6458 14.41
  Grandmaster  4745 10.59
      Diamond   718  1.60
     Unranked   224  0.50
     Platinum   131  0.29
         Gold   119  0.27
       Bronze    73  0.16
       Silver     9  0.02
                  2  0.00

=== region ===
 region   cnt   pct
 Europe 21022 46.91
     US 12699 28.34
Unknown  5748 12.83
  Korea  3604  8.04
  China  1742  3.89
            2  0.00

=== realm ===
        realm   cnt   pct
       Europe 20777 46.36
North America 12490 27.87
      Unknown  5748 12.83
        Korea  2835  6.33
        Ch

### isInClan distribution

In [16]:
IS_IN_CLAN_SQL = """\
SELECT
    isInClan,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM replay_players_raw
GROUP BY isInClan
ORDER BY cnt DESC
"""
is_in_clan_df = con.execute(IS_IN_CLAN_SQL).df()
print("=== isInClan distribution ===")
print(is_in_clan_df.to_string(index=False))

=== isInClan distribution ===
 isInClan   cnt  pct
    False 33210 74.1
     True 11607 25.9


### clanTag top-20

In [17]:
CLAN_TAG_TOP20_SQL = """\
SELECT
    clanTag,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
    ROUND(100.0 * COUNT(*) / (
        SELECT COUNT(*) FROM replay_players_raw
        WHERE clanTag != ''
    ), 2) AS pct_of_non_empty
FROM replay_players_raw
WHERE clanTag != ''
GROUP BY clanTag
ORDER BY cnt DESC
LIMIT 20
"""
clan_tag_top20_df = con.execute(CLAN_TAG_TOP20_SQL).df()
print("=== clanTag top-20 (non-empty) ===")
print(clan_tag_top20_df.to_string(index=False))

=== clanTag top-20 (non-empty) ===
clanTag  cnt  pct_of_total  pct_of_non_empty
     αX  778          6.70              6.70
 PSISTM  532          4.58              4.58
   mouz  513          4.42              4.42
   RBLN  466          4.01              4.01
  QLASH  454          3.91              3.91
   ENCE  443          3.82              3.82
   Ex0n  322          2.77              2.77
  Kaizi  219          1.89              1.89
   mlem  214          1.84              1.84
   xkom  212          1.83              1.83
 TeamNV  197          1.70              1.70
   сSсǃ  188          1.62              1.62
  Mkers  179          1.54              1.54
  IxGeu  170          1.46              1.46
   ROOT  162          1.40              1.40
   Ting  160          1.38              1.38
 BASKGG  157          1.35              1.35
  ASTsc  139          1.20              1.20
 St0rmG  138          1.19              1.19
 GGGSC2  128          1.10              1.10


In [18]:
# replays_meta_raw STRUCT categorical fields
META_CAT_COLS = [
    "game_speed", "game_speed_init", "map_name",
    "game_version_meta", "base_build", "data_build",
]

meta_cat_results = {}
for col in META_CAT_COLS:
    if col == "map_name":
        sql = f"""
        SELECT {col}, COUNT(*) AS cnt
        FROM struct_flat
        GROUP BY {col}
        ORDER BY cnt DESC
        LIMIT 20
        """
    else:
        sql = f"""
        SELECT {col}, COUNT(*) AS cnt,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
        FROM struct_flat
        GROUP BY {col}
        ORDER BY cnt DESC
        """
    # struct_flat is a pandas df, so register it for SQL access
    df = con.execute(sql).df()
    meta_cat_results[col] = df
    print(f"\n=== {col} ===")
    if col == "map_name":
        print(f"(top 20 of {con.execute(f'SELECT COUNT(DISTINCT {col}) FROM struct_flat').fetchone()[0]} distinct)")
    print(df.to_string(index=False))


=== game_speed ===
game_speed   cnt   pct
    Faster 22390 100.0

=== game_speed_init ===
game_speed_init   cnt   pct
         Faster 22390 100.0

=== map_name ===
(top 20 of 188 distinct)
           map_name  cnt
2000 Atmospheres LE  966
      Acid Plant LE  667
        Catalyst LE  586
           Oxide LE  570
     Romanticide LE  568
  Eternal Empire LE  528
     King's Cove LE  509
      Lightshade LE  508
 Kairos Junction LE  496
    Abyssal Reef LE  481
      Ever Dream LE  471
  Lost and Found LE  461
      Jagannatha LE  440
       Deathaura LE  438
       Acropolis LE  419
 Pillars of Gold LE  390
       [ESL] Data-C  387
 Port Aleksander LE  381
       Blackpink LE  370
    Cyber Forest LE  364

=== game_version_meta ===
game_version_meta  cnt   pct
      5.0.7.84643 2511 11.21
      5.0.9.87702 1094  4.89
      5.0.8.86383 1071  4.78
     5.0.10.88500  988  4.41
      4.8.3.72282  933  4.17
     5.0.10.89165  865  3.86
     5.0.12.91115  859  3.84
     5.0.11.90136  820  3.

In [19]:
# Game speed assertion -- MUST pass before Section E
speed_counts = con.execute(
    "SELECT game_speed, COUNT(*) AS cnt FROM struct_flat GROUP BY game_speed ORDER BY cnt DESC"
).df()
print(speed_counts)
assert set(speed_counts["game_speed"].dropna()) == {"Faster"}, (
    f"Expected only 'Faster' game speed; found: {speed_counts['game_speed'].unique()}"
)
print("All replays confirmed Faster speed -- duration conversion is safe.")

  game_speed    cnt
0     Faster  22390
All replays confirmed Faster speed -- duration conversion is safe.


In [20]:
# Visualizations: bar charts for categorical fields
# result bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(result_dist["result"].astype(str), result_dist["cnt"])
ax.set_title("Result Value Distribution")
ax.set_xlabel("result")
ax.set_ylabel("Count")
for i, (val, cnt) in enumerate(zip(result_dist["result"], result_dist["cnt"])):
    ax.text(i, cnt, str(cnt), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_result_bar.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/1710186016.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# race bar chart
race_df = rp_cat_results["race"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(race_df["race"].astype(str), race_df["cnt"])
ax.set_title("Race Distribution")
ax.set_xlabel("race")
ax.set_ylabel("Count")
for i, (val, cnt) in enumerate(zip(race_df["race"], race_df["cnt"])):
    ax.text(i, cnt, str(cnt), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_race_bar.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/2871972279.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# selectedRace bar chart
sr_df = rp_cat_results["selectedRace"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sr_df["selectedRace"].astype(str), sr_df["cnt"])
ax.set_title("Selected Race Distribution")
ax.set_xlabel("selectedRace")
ax.set_ylabel("Count")
for i, (val, cnt) in enumerate(zip(sr_df["selectedRace"], sr_df["cnt"])):
    ax.text(i, cnt, str(cnt), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_selectedrace_bar.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/2017757904.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# highestLeague bar chart
hl_df = rp_cat_results["highestLeague"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hl_df["highestLeague"].astype(str), hl_df["cnt"])
ax.set_title("Highest League Distribution")
ax.set_xlabel("highestLeague")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
for i, (val, cnt) in enumerate(zip(hl_df["highestLeague"], hl_df["cnt"])):
    ax.text(i, cnt, str(cnt), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_highestleague_bar.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/1461461266.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# region bar chart
reg_df = rp_cat_results["region"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(reg_df["region"].astype(str), reg_df["cnt"])
ax.set_title("Region Distribution")
ax.set_xlabel("region")
ax.set_ylabel("Count")
for i, (val, cnt) in enumerate(zip(reg_df["region"], reg_df["cnt"])):
    ax.text(i, cnt, str(cnt), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_region_bar.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/1026911534.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section E: Numeric Descriptive Statistics

**Prerequisite:** Section D game speed assertion passed.
The game-loop-to-seconds conversion uses `/ 22.4` (22.4 game loops/second:
standard Faster speed = 16 base loops/second x 1.4 Faster multiplier).

**In-game field distinction:** APM, SQ, and supplyCappedPercent are
in-game-only fields (computed from replay actions, available only
post-game). Their presence in the EDA does not imply pre-game
availability. This distinction will be formally enforced at the feature
engineering stage (Phase 03).

In [25]:
# Descriptive stats for replay_players_raw numeric columns
RP_NUM_COLS = ["MMR", "APM", "SQ", "supplyCappedPercent", "handicap"]
rp_num_stats = {}

for col in RP_NUM_COLS:
    sql = f"""
    SELECT
        MIN({col}) AS min_val,
        MAX({col}) AS max_val,
        ROUND(AVG({col}), 2) AS mean_val,
        ROUND(MEDIAN({col}), 2) AS median_val,
        ROUND(STDDEV({col}), 2) AS stddev_val,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY {col}) AS p05,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY {col}) AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY {col}) AS p75,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {col}) AS p95
    FROM replay_players_raw
    WHERE {col} IS NOT NULL
    """
    df = con.execute(sql).df()
    rp_num_stats[col] = df
    print(f"\n=== {col} ===")
    print(df.to_string(index=False))


=== MMR ===
 min_val  max_val  mean_val  median_val  stddev_val  p05  p25  p75    p95
  -36400     7464    738.71         0.0     3035.53  0.0  0.0  0.0 6493.0

=== APM ===
 min_val  max_val  mean_val  median_val  stddev_val   p05   p25   p75   p95
       0     1248    355.57       349.0      104.87 219.0 303.0 410.0 523.0

=== SQ ===
    min_val  max_val  mean_val  median_val  stddev_val  p05   p25   p75   p95
-2147483648      187 -95711.06       123.0 14345597.84 91.0 110.0 136.0 152.0

=== supplyCappedPercent ===
 min_val  max_val  mean_val  median_val  stddev_val  p05  p25  p75  p95
       0      100      7.24         6.0        4.71  2.0  4.0  9.0 16.0

=== handicap ===
 min_val  max_val  mean_val  median_val  stddev_val   p05   p25   p75   p95
       0      100     100.0       100.0        0.67 100.0 100.0 100.0 100.0


### SQ stats excluding INT32_MIN sentinel

In [26]:
SQ_NO_SENTINEL_SQL = """\
SELECT
    MIN(SQ) AS min_val,
    MAX(SQ) AS max_val,
    ROUND(AVG(SQ), 2) AS mean_val,
    ROUND(MEDIAN(SQ), 2) AS median_val,
    ROUND(STDDEV(SQ), 2) AS stddev_val,
    PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY SQ) AS p05,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY SQ) AS p25,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY SQ) AS p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY SQ) AS p95,
    COUNT(*) AS n_rows
FROM replay_players_raw
WHERE SQ IS NOT NULL AND SQ != -2147483648
"""
sq_no_sentinel = con.execute(SQ_NO_SENTINEL_SQL).df()
print("=== SQ stats (sentinel excluded) ===")
print(sq_no_sentinel.to_string(index=False))
print(
    "\nThe main SQ descriptive statistics (Section E) are contaminated by 2 rows "
    "containing the INT32_MIN sentinel value (-2147483648). The stats above "
    "exclude those rows. The sentinel causes the main SQ mean and stddev to be "
    "misleading. Refer to the sentinel-excluded stats above for the clean SQ "
    "distribution's actual median and range."
)

=== SQ stats (sentinel excluded) ===
 min_val  max_val  mean_val  median_val  stddev_val  p05   p25   p75   p95  n_rows
     -51      187    122.38       123.0       18.91 91.0 110.0 136.0 152.0   44815

The main SQ descriptive statistics (Section E) are contaminated by 2 rows containing the INT32_MIN sentinel value (-2147483648). The stats above exclude those rows. The sentinel causes the main SQ mean and stddev to be misleading. Refer to the sentinel-excluded stats above for the clean SQ distribution's actual median and range.


In [27]:
# Positional columns: cardinality and ranges
POS_COLS = ["startDir", "startLocX", "startLocY"]

for col in POS_COLS:
    sql = f"""
    SELECT
        COUNT(DISTINCT {col}) AS cardinality,
        MIN({col}) AS min_val,
        MAX({col}) AS max_val
    FROM replay_players_raw
    WHERE {col} IS NOT NULL
    """
    df = con.execute(sql).df()
    print(f"\n=== {col} ===")
    print(df.to_string(index=False))


=== startDir ===
 cardinality  min_val  max_val
          13        0       12

=== startLocX ===
 cardinality  min_val  max_val
         117        0      211

=== startLocY ===
 cardinality  min_val  max_val
         115        0      207


In [28]:
# Duration stats from replays_meta_raw (elapsed_game_loops)
DURATION_STATS_SQL = """\
SELECT
    MIN(elapsed_game_loops) AS min_val,
    MAX(elapsed_game_loops) AS max_val,
    ROUND(AVG(elapsed_game_loops), 2) AS mean_val,
    ROUND(MEDIAN(elapsed_game_loops), 2) AS median_val,
    ROUND(STDDEV(elapsed_game_loops), 2) AS stddev_val,
    PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY elapsed_game_loops) AS p05,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY elapsed_game_loops) AS p25,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY elapsed_game_loops) AS p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY elapsed_game_loops) AS p95,
    ROUND(MIN(elapsed_game_loops) / 22.4, 2) AS min_seconds,
    ROUND(MAX(elapsed_game_loops) / 22.4, 2) AS max_seconds,
    ROUND(AVG(elapsed_game_loops) / 22.4, 2) AS mean_seconds,
    ROUND(MEDIAN(elapsed_game_loops) / 22.4, 2) AS median_seconds
FROM struct_flat
WHERE elapsed_game_loops IS NOT NULL
"""
duration_stats = con.execute(DURATION_STATS_SQL).df()
print("=== elapsed_game_loops (+ seconds conversion at 22.4 loops/sec) ===")
print(duration_stats.to_string(index=False))

=== elapsed_game_loops (+ seconds conversion at 22.4 loops/sec) ===
 min_val  max_val  mean_val  median_val  stddev_val     p05      p25      p75     p95  min_seconds  max_seconds  mean_seconds  median_seconds
      12   136028  16105.26     14580.0     7795.59 6640.45 11111.25 19390.75 30270.1         0.54      6072.68        718.98          650.89


In [29]:
# Map dimensions
MAP_DIM_SQL = """\
SELECT
    map_size_x, map_size_y, COUNT(*) AS cnt
FROM struct_flat
GROUP BY map_size_x, map_size_y
ORDER BY cnt DESC
"""
map_dims = con.execute(MAP_DIM_SQL).df()
print("=== Map dimensions (map_size_x, map_size_y) ===")
print(map_dims.to_string(index=False))

=== Map dimensions (map_size_x, map_size_y) ===
 map_size_x  map_size_y  cnt
        168         184 1624
        176         176 1615
        176         184 1052
        160         168 1030
        224         224  966
        192         208  886
        216         216  868
        168         168  858
        184         184  800
        200         176  783
        152         168  765
        200         184  754
        184         176  727
        168         176  617
        192         176  612
        200         192  583
        192         192  528
        200         216  471
        184         168  465
        144         160  459
        200         200  449
        168         200  441
        184         144  394
        176         160  384
        192         168  362
        176         152  309
        192         184  308
        208         208  304
        184         192  287
        216         192  276
          0           0  273
        168         160 

In [30]:
# Verification: MMR data before plot
mmr_data = con.execute(
    "SELECT MMR FROM replay_players_raw WHERE MMR IS NOT NULL"
).df()
print(f"=== MMR data for plot ({len(mmr_data)} rows) ===")
print(mmr_data.describe().to_string())

=== MMR data for plot (44817 rows) ===
                MMR
count  44817.000000
mean     738.714506
std     3035.530130
min   -36400.000000
25%        0.000000
50%        0.000000
75%        0.000000
max     7464.000000


In [31]:
# Histogram: MMR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(mmr_data["MMR"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("MMR Distribution (histogram)")
axes[0].set_xlabel("MMR")
axes[0].set_ylabel("Frequency")
axes[1].boxplot(mmr_data["MMR"], vert=True)
axes[1].set_title("MMR Distribution (boxplot)")
axes[1].set_ylabel("MMR")
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_mmr_histogram.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/3483500727.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# Verification: APM data before plot
apm_data = con.execute(
    "SELECT APM FROM replay_players_raw WHERE APM IS NOT NULL"
).df()
print(f"=== APM data for plot ({len(apm_data)} rows) ===")
print(apm_data.describe().to_string())

=== APM data for plot (44817 rows) ===
                APM
count  44817.000000
mean     355.565656
std      104.872041
min        0.000000
25%      303.000000
50%      349.000000
75%      410.000000
max     1248.000000


In [33]:
# Histogram: APM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(apm_data["APM"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("APM Distribution (histogram)")
axes[0].set_xlabel("APM")
axes[0].set_ylabel("Frequency")
axes[1].boxplot(apm_data["APM"], vert=True)
axes[1].set_title("APM Distribution (boxplot)")
axes[1].set_ylabel("APM")
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_apm_histogram.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/1495555319.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
# Verification: SQ data before plot
sq_data = con.execute(
    "SELECT SQ FROM replay_players_raw WHERE SQ IS NOT NULL"
).df()
print(f"=== SQ data for plot ({len(sq_data)} rows) ===")
print(sq_data.describe().to_string())

=== SQ data for plot (44817 rows) ===
                 SQ
count  4.481700e+04
mean  -9.571106e+04
std    1.434560e+07
min   -2.147484e+09
25%    1.100000e+02
50%    1.230000e+02
75%    1.360000e+02
max    1.870000e+02


In [35]:
# Histogram: SQ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sq_data["SQ"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("SQ Distribution (histogram)")
axes[0].set_xlabel("SQ")
axes[0].set_ylabel("Frequency")
axes[1].boxplot(sq_data["SQ"], vert=True)
axes[1].set_title("SQ Distribution (boxplot)")
axes[1].set_ylabel("SQ")
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_sq_histogram.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/515430424.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
# Verification: supplyCappedPercent data before plot
sc_data = con.execute(
    "SELECT supplyCappedPercent FROM replay_players_raw WHERE supplyCappedPercent IS NOT NULL"
).df()
print(f"=== supplyCappedPercent data for plot ({len(sc_data)} rows) ===")
print(sc_data.describe().to_string())

=== supplyCappedPercent data for plot (44817 rows) ===
       supplyCappedPercent
count         44817.000000
mean              7.239931
std               4.712071
min               0.000000
25%               4.000000
50%               6.000000
75%               9.000000
max             100.000000


In [37]:
# Histogram: supplyCappedPercent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sc_data["supplyCappedPercent"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("supplyCappedPercent Distribution (histogram)")
axes[0].set_xlabel("supplyCappedPercent")
axes[0].set_ylabel("Frequency")
axes[1].boxplot(sc_data["supplyCappedPercent"], vert=True)
axes[1].set_title("supplyCappedPercent Distribution (boxplot)")
axes[1].set_ylabel("supplyCappedPercent")
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_supplycapped_histogram.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/1300599072.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# Verification: Duration data before plot
dur_data = con.execute(
    "SELECT elapsed_game_loops / 22.4 / 60.0 AS duration_min FROM struct_flat WHERE elapsed_game_loops IS NOT NULL"
).df()
print(f"=== Duration data for plot ({len(dur_data)} rows) ===")
print(dur_data.describe().to_string())

=== Duration data for plot (22390 rows) ===
       duration_min
count  22390.000000
mean      11.983080
std        5.800294
min        0.008929
25%        8.267299
50%       10.848214
75%       14.427641
max      101.211310


In [39]:
# Histogram: Duration in minutes
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dur_data["duration_min"], bins=60, edgecolor="black", alpha=0.7)
ax.set_title("Game Duration Distribution (minutes)")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Frequency")
ax.axvline(dur_data["duration_min"].median(), color="red", ls="--", label=f"Median: {dur_data['duration_min'].median():.1f} min")
ax.legend()
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_duration_histogram.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/3441009469.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section E1: Skewness and Kurtosis (EDA Manual 3.1)

In [40]:
SKEW_KURT_COLS = [
    "MMR", "APM", "SQ", "supplyCappedPercent", "handicap",
    "startDir", "startLocX", "startLocY",
    "color_a", "color_b", "color_g", "color_r",
    "playerID", "userID",
]
skew_kurt_results = {}

for col in SKEW_KURT_COLS:
    sql = f"""
    SELECT
        ROUND(SKEWNESS({col}), 4) AS skewness,
        ROUND(KURTOSIS({col}), 4) AS kurtosis
    FROM replay_players_raw
    WHERE {col} IS NOT NULL
    """
    row = con.execute(sql).fetchone()
    skew_kurt_results[col] = {"skewness": row[0], "kurtosis": row[1]}

# elapsed_game_loops from struct_flat
_sk_row = con.execute("""
    SELECT
        ROUND(SKEWNESS(elapsed_game_loops), 4) AS skewness,
        ROUND(KURTOSIS(elapsed_game_loops), 4) AS kurtosis
    FROM struct_flat
    WHERE elapsed_game_loops IS NOT NULL
""").fetchone()
skew_kurt_results["elapsed_game_loops"] = {"skewness": _sk_row[0], "kurtosis": _sk_row[1]}

skew_kurt_df = pd.DataFrame(
    [{"column": k, **v} for k, v in skew_kurt_results.items()]
)
print("=== Skewness and Kurtosis ===")
print(skew_kurt_df.to_string(index=False))

=== Skewness and Kurtosis ===
             column  skewness   kurtosis
                MMR   -5.7590    77.8914
                APM   -0.2024     3.6247
                 SQ -149.6897 22405.9998
supplyCappedPercent    2.2456    18.1756
           handicap -149.6897 22405.9998
           startDir   -0.0206    -1.3235
          startLocX    0.0563    -1.7766
          startLocY    0.0527    -1.7663
            color_a -149.6897 22405.9998
            color_b    0.0025    -1.9367
            color_g    2.1054     7.8214
            color_r    0.0429    -1.9807
           playerID    0.6305     5.2759
             userID    1.2019     1.6383
 elapsed_game_loops    2.0331    10.6597


## Section E2: Zero Counts (EDA Manual 3.1)

Count exact-zero values for each numeric column in `replay_players_raw`
and the INT32_MIN sentinel for SQ (value = -2147483648).
Also count exact-zero `elapsed_game_loops` in `replays_meta_raw`.

In [41]:
ZERO_COUNT_SQL = """\
SELECT
    COUNT(*) FILTER (WHERE MMR = 0) AS MMR_zero,
    COUNT(*) FILTER (WHERE APM = 0) AS APM_zero,
    COUNT(*) FILTER (WHERE SQ = 0) AS SQ_zero,
    COUNT(*) FILTER (WHERE SQ = -2147483648) AS SQ_sentinel,
    COUNT(*) FILTER (WHERE supplyCappedPercent = 0) AS supplyCappedPercent_zero,
    COUNT(*) FILTER (WHERE handicap = 0) AS handicap_zero,
    COUNT(*) FILTER (WHERE startDir = 0) AS startDir_zero,
    COUNT(*) FILTER (WHERE startLocX = 0) AS startLocX_zero,
    COUNT(*) FILTER (WHERE startLocY = 0) AS startLocY_zero,
    COUNT(*) FILTER (WHERE color_a = 0) AS color_a_zero,
    COUNT(*) FILTER (WHERE color_b = 0) AS color_b_zero,
    COUNT(*) FILTER (WHERE color_g = 0) AS color_g_zero,
    COUNT(*) FILTER (WHERE color_r = 0) AS color_r_zero,
    COUNT(*) FILTER (WHERE playerID = 0) AS playerID_zero,
    COUNT(*) FILTER (WHERE userID = 0) AS userID_zero
FROM replay_players_raw
"""
zero_counts = con.execute(ZERO_COUNT_SQL).df()
print("=== Zero counts (replay_players_raw) ===")
print(zero_counts.to_string(index=False))

=== Zero counts (replay_players_raw) ===
 MMR_zero  APM_zero  SQ_zero  SQ_sentinel  supplyCappedPercent_zero  handicap_zero  startDir_zero  startLocX_zero  startLocY_zero  color_a_zero  color_b_zero  color_g_zero  color_r_zero  playerID_zero  userID_zero
    37489      1132        0            2                       298              2              5               5               5             2         21191           547           530              0         8611


In [42]:
DURATION_ZERO_COUNT_SQL = """\
SELECT
    COUNT(*) FILTER (WHERE elapsed_game_loops = 0) AS elapsed_game_loops_zero
FROM struct_flat
"""
duration_zero_counts = con.execute(DURATION_ZERO_COUNT_SQL).df()
print("=== Zero counts (elapsed_game_loops) ===")
print(duration_zero_counts.to_string(index=False))

=== Zero counts (elapsed_game_loops) ===
 elapsed_game_loops_zero
                       0


### MMR zero-spike interpretation

In [43]:
MMR_ZERO_BY_RESULT_SQL = """\
SELECT
    result,
    COUNT(*) FILTER (WHERE MMR = 0) AS mmr_zero_cnt,
    COUNT(*) AS total_cnt,
    ROUND(100.0 * COUNT(*) FILTER (WHERE MMR = 0) / COUNT(*), 2)
        AS mmr_zero_pct
FROM replay_players_raw
GROUP BY result
ORDER BY total_cnt DESC
"""
mmr_zero_by_result = con.execute(MMR_ZERO_BY_RESULT_SQL).df()
print("=== MMR=0 by result ===")
print(mmr_zero_by_result.to_string(index=False))

=== MMR=0 by result ===
   result  mmr_zero_cnt  total_cnt  mmr_zero_pct
     Loss         18597      22409         82.99
      Win         18876      22382         84.34
Undecided            14         24         58.33
      Tie             2          2        100.00


In [44]:
MMR_ZERO_BY_LEAGUE_SQL = """\
SELECT
    highestLeague,
    COUNT(*) FILTER (WHERE MMR = 0) AS mmr_zero_cnt,
    COUNT(*) AS total_cnt,
    ROUND(100.0 * COUNT(*) FILTER (WHERE MMR = 0) / COUNT(*), 2)
        AS mmr_zero_pct
FROM replay_players_raw
GROUP BY highestLeague
ORDER BY total_cnt DESC
"""
mmr_zero_by_league = con.execute(MMR_ZERO_BY_LEAGUE_SQL).df()
print("=== MMR=0 by highestLeague ===")
print(mmr_zero_by_league.to_string(index=False))
# Interpretation: if MMR=0 rate is uniform across all result categories,
# that supports the "not reported" hypothesis over a "loss-correlated sentinel" hypothesis.
_mmr_zero_pcts = mmr_zero_by_result["mmr_zero_pct"].dropna()
_mmr_zero_spread = round(float(_mmr_zero_pcts.max() - _mmr_zero_pcts.min()), 2)
print(
    f"\nMMR=0 rate spread across result categories: {_mmr_zero_spread} percentage points. "
    + (
        "Small spread supports 'sentinel = not reported' hypothesis."
        if _mmr_zero_spread < 5.0
        else "Large spread suggests possible correlation with outcome."
    )
)

=== MMR=0 by highestLeague ===
highestLeague  mmr_zero_cnt  total_cnt  mmr_zero_pct
      Unknown         30116      32338         93.13
       Master          3743       6458         57.96
  Grandmaster          2860       4745         60.27
      Diamond           376        718         52.37
     Unranked           224        224        100.00
     Platinum            60        131         45.80
         Gold            64        119         53.78
       Bronze            41         73         56.16
       Silver             3          9         33.33
                          2          2        100.00

MMR=0 rate spread across result categories: 41.67 percentage points. Large spread suggests possible correlation with outcome.


In [45]:
DUPLICATE_CHECK_SQL = """\
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT filename || '|' || toon_id) AS distinct_player_rows,
    COUNT(*) - COUNT(DISTINCT filename || '|' || toon_id) AS duplicate_rows
FROM replay_players_raw
"""
duplicate_check = con.execute(DUPLICATE_CHECK_SQL).df()
print("=== Duplicate check (replay_players_raw) ===")
print(duplicate_check.to_string(index=False))

=== Duplicate check (replay_players_raw) ===
 total_rows  distinct_player_rows  duplicate_rows
      44817                 44817               0


## Section F: Temporal Range

Use string-based MIN/MAX -- ISO 8601 strings sort lexicographically.
No STRPTIME: the format `2016-07-29T04:50:12.5655603Z` contains
7-digit fractional seconds (.NET precision) that `%f` (6-digit) cannot handle.

In [46]:
TEMPORAL_RANGE_SQL = """\
SELECT
    MIN(details.timeUTC) AS earliest,
    MAX(details.timeUTC) AS latest,
    COUNT(DISTINCT SUBSTR(details.timeUTC, 1, 7)) AS distinct_months
FROM replays_meta_raw
"""
temporal_range = con.execute(TEMPORAL_RANGE_SQL).df()
print("=== Temporal range ===")
print(temporal_range.to_string(index=False))

=== Temporal range ===
                earliest                       latest  distinct_months
2016-01-07T02:21:46.002Z 2024-12-01T23:48:45.2511615Z               76


In [47]:
# Time-series: replay counts by month
MONTHLY_COUNTS_SQL = """\
SELECT
    SUBSTR(details.timeUTC, 1, 7) AS month,
    COUNT(*) AS cnt
FROM replays_meta_raw
GROUP BY month
ORDER BY month
"""
monthly = con.execute(MONTHLY_COUNTS_SQL).df()
print("=== Monthly replay counts ===")
print(monthly.to_string(index=False))

=== Monthly replay counts ===
  month  cnt
2016-01  139
2016-02  256
2016-03  100
2016-07   60
2017-02  222
2017-03  187
2017-04  391
2017-05   14
2017-06  332
2017-07  188
2017-09  236
2017-10   30
2017-11  399
2018-01  420
2018-02  315
2018-03  356
2018-06  709
2018-07  450
2018-09  442
2018-10   44
2018-11  380
2018-12   64
2019-01    2
2019-02  612
2019-03  562
2019-04   46
2019-05  540
2019-06  330
2019-07  596
2019-08  169
2019-09  601
2019-10   82
2019-11  346
2020-02  423
2020-03   15
2020-04  172
2020-05  157
2020-06  547
2020-07  501
2020-08  250
2020-09  549
2020-11  107
2020-12  121
2021-01  157
2021-02  261
2021-03  183
2021-05  587
2021-06  258
2021-07  297
2021-08  495
2021-09  272
2021-10  812
2021-11  143
2021-12  371
2022-01   84
2022-02  263
2022-05  592
2022-06  107
2022-07  541
2022-08   68
2022-09  212
2022-10  562
2022-11  580
2022-12  285
2023-02  260
2023-05  441
2023-06  257
2023-08  122
2023-12  672
2024-02  187
2024-05   72
2024-06  367
2024-08  154
2024-09 

In [48]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(monthly)), monthly["cnt"], tick_label=monthly["month"])
ax.set_title("Replays Over Time (by month)")
ax.set_xlabel("Month")
ax.set_ylabel("Replay Count")
ax.tick_params(axis="x", rotation=90, labelsize=7)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_04_replays_over_time.png", dpi=150)
plt.show()
plt.close()

/var/folders/xr/5wlm64ks7_lf9yzgpxwy7f7w0000gn/T/ipykernel_78797/2065672820.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section G: Error Column Census

In [49]:
ERROR_CENSUS_SQL = """\
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE gameEventsErr = TRUE) AS game_err,
    COUNT(*) FILTER (WHERE messageEventsErr = TRUE) AS msg_err,
    COUNT(*) FILTER (WHERE trackerEvtsErr = TRUE) AS tracker_err
FROM replays_meta_raw
"""
error_census = con.execute(ERROR_CENSUS_SQL).df()
print("=== Error column census ===")
print(error_census.to_string(index=False))

=== Error column census ===
 total  game_err  msg_err  tracker_err
 22390         0        0            0


## Section H: Dead/Constant/Near-Constant Field Detection

Cardinality and uniqueness ratio (`COUNT(DISTINCT col) / COUNT(*)`)
for all columns. Flag cardinality = 1 (constant) or uniqueness
ratio < 0.001 (near-constant). Denominator is `COUNT(*)` (all rows,
including NULLs) by design.

In [50]:
# replay_players_raw cardinality
RP_ALL_COLS = [
    "filename", "toon_id", "nickname", "playerID", "userID",
    "isInClan", "clanTag", "MMR", "race", "selectedRace",
    "handicap", "region", "realm", "highestLeague", "result",
    "APM", "SQ", "supplyCappedPercent", "startDir", "startLocX",
    "startLocY", "color_a", "color_b", "color_g", "color_r",
]

rp_card_records = []
for col in RP_ALL_COLS:
    sql = f"""
    SELECT
        COUNT(DISTINCT {col}) AS cardinality,
        COUNT(*) AS total_rows,
        ROUND(COUNT(DISTINCT {col})::DOUBLE / COUNT(*)::DOUBLE, 6) AS uniqueness_ratio
    FROM replay_players_raw
    """
    row = con.execute(sql).fetchone()
    rp_card_records.append({
        "table": "replay_players_raw",
        "column": col,
        "cardinality": row[0],
        "total_rows": row[1],
        "uniqueness_ratio": row[2],
    })

rp_card_df = pd.DataFrame(rp_card_records)
print("=== replay_players_raw cardinality ===")
print(rp_card_df.to_string(index=False))

=== replay_players_raw cardinality ===
             table              column  cardinality  total_rows  uniqueness_ratio
replay_players_raw            filename        22390       44817          0.499587
replay_players_raw             toon_id         2495       44817          0.055671
replay_players_raw            nickname         1106       44817          0.024678
replay_players_raw            playerID            9       44817          0.000201
replay_players_raw              userID           16       44817          0.000357
replay_players_raw            isInClan            2       44817          0.000045
replay_players_raw             clanTag          257       44817          0.005734
replay_players_raw                 MMR         1031       44817          0.023005
replay_players_raw                race            6       44817          0.000134
replay_players_raw        selectedRace            8       44817          0.000179
replay_players_raw            handicap            2       4

In [51]:
# struct_flat (replays_meta_raw) cardinality
SF_COLS = [
    "game_speed", "is_blizzard_map", "time_utc",
    "elapsed_game_loops", "game_version_header",
    "base_build", "data_build", "game_version_meta",
    "map_name", "max_players", "game_speed_init",
    "is_blizzard_map_init", "map_size_x", "map_size_y",
    "gameEventsErr", "messageEventsErr", "trackerEvtsErr", "filename",
]

sf_card_records = []
for col in SF_COLS:
    sql = f"""
    SELECT
        COUNT(DISTINCT {col}) AS cardinality,
        COUNT(*) AS total_rows,
        ROUND(COUNT(DISTINCT {col})::DOUBLE / COUNT(*)::DOUBLE, 6) AS uniqueness_ratio
    FROM struct_flat
    """
    row = con.execute(sql).fetchone()
    sf_card_records.append({
        "table": "replays_meta_raw (struct_flat)",
        "column": col,
        "cardinality": row[0],
        "total_rows": row[1],
        "uniqueness_ratio": row[2],
    })

sf_card_df = pd.DataFrame(sf_card_records)
print("=== struct_flat (replays_meta_raw) cardinality ===")
print(sf_card_df.to_string(index=False))

=== struct_flat (replays_meta_raw) cardinality ===
                         table               column  cardinality  total_rows  uniqueness_ratio
replays_meta_raw (struct_flat)           game_speed            1       22390          0.000045
replays_meta_raw (struct_flat)      is_blizzard_map            2       22390          0.000089
replays_meta_raw (struct_flat)             time_utc        22344       22390          0.997946
replays_meta_raw (struct_flat)   elapsed_game_loops        14045       22390          0.627289
replays_meta_raw (struct_flat)  game_version_header           46       22390          0.002054
replays_meta_raw (struct_flat)           base_build           46       22390          0.002054
replays_meta_raw (struct_flat)           data_build           46       22390          0.002054
replays_meta_raw (struct_flat)    game_version_meta           46       22390          0.002054
replays_meta_raw (struct_flat)             map_name          188       22390          0.008397

In [52]:
# Combine and flag dead/constant/near-constant
all_card_df = pd.concat([rp_card_df, sf_card_df], ignore_index=True)

flagged = all_card_df[
    (all_card_df["cardinality"] == 1)
    | (all_card_df["uniqueness_ratio"] < 0.001)
].copy()
flagged["flag"] = "near-constant"
flagged.loc[flagged["cardinality"] == 1, "flag"] = "constant"
print("=== Flagged dead/constant/near-constant columns ===")
if len(flagged) > 0:
    print(flagged.to_string(index=False))
else:
    print("(none)")

=== Flagged dead/constant/near-constant columns ===
                         table               column  cardinality  total_rows  uniqueness_ratio          flag
            replay_players_raw             playerID            9       44817          0.000201 near-constant
            replay_players_raw               userID           16       44817          0.000357 near-constant
            replay_players_raw             isInClan            2       44817          0.000045 near-constant
            replay_players_raw                 race            6       44817          0.000134 near-constant
            replay_players_raw         selectedRace            8       44817          0.000179 near-constant
            replay_players_raw             handicap            2       44817          0.000045 near-constant
            replay_players_raw               region            6       44817          0.000134 near-constant
            replay_players_raw                realm            9       44817

**Threshold sensitivity note:** The uniqueness_ratio < 0.001 threshold
is appropriate for this dataset (N=44,817 rows). For larger datasets
(N > 1M), the same threshold would flag more columns as near-constant
because uniqueness_ratio = cardinality / N decreases with N even for
columns with stable cardinality. Re-evaluate the threshold for each
dataset based on its row count and the cardinality distribution.

In [53]:
# Field classification: pre-game / in-game / post-game / identifier / target / constant
# Covers all 25 replay_players_raw columns and all struct_flat-extracted fields.
# Purpose: downstream steps use this dict to select features without parsing prose.
FIELD_CLASSIFICATION = {
    "replay_players_raw": {
        "identifier": ["filename", "toon_id", "nickname", "playerID", "userID"],
        "pre_game": [
            "MMR", "race", "selectedRace", "handicap", "region", "realm",
            "highestLeague", "isInClan", "clanTag", "startDir", "startLocX",
            "startLocY", "color_a", "color_b", "color_g", "color_r",
        ],
        "in_game": ["APM", "SQ", "supplyCappedPercent"],
        "target": ["result"],
    },
    "replays_meta_raw_struct_flat": {
        "identifier": ["filename"],
        "pre_game": [
            "time_utc", "game_version_header", "base_build", "data_build",
            "game_version_meta", "map_name", "max_players", "map_size_x",
            "map_size_y", "is_blizzard_map", "is_blizzard_map_init",
        ],
        "in_game": [],  # APM, SQ, supplyCappedPercent are in replay_players_raw, not here
        "post_game": ["elapsed_game_loops"],  # total match duration; only known after match ends (same as AoE2 duration_sec)
        "constant": [
            "game_speed", "game_speed_init", "gameEventsErr",
            "messageEventsErr", "trackerEvtsErr",
        ],
    },
    "classification_notes": {
        "pre_game": (
            "Available before match start. Usable as prediction features "
            "without temporal leakage."
        ),
        "in_game": (
            "Computed from replay actions; available only post-game. Using these "
            "as features requires in-game prediction framing (Phase 02 decision)."
        ),
        "post_game": (
            "Terminal match metadata knowable only after the result is set. "
            "elapsed_game_loops (total match duration) is only defined once the match "
            "concludes — unlike APM/SQ which are computed incrementally from action "
            "streams. Post-game scalars are excluded from all predictive feature sets "
            "(Invariant #3)."
        ),
        "identifier": "Player/replay identity columns. Not features.",
        "target": "Prediction target. Never a feature.",
        "constant": "Single-value columns. No predictive information.",
    },
}
# Sanity check: replay_players_raw total column count must be 25
_rp_total = sum(
    len(v)
    for k, v in FIELD_CLASSIFICATION["replay_players_raw"].items()
)
assert _rp_total == 25, (
    f"replay_players_raw column count mismatch: expected 25, got {_rp_total}"
)
print(f"FIELD_CLASSIFICATION replay_players_raw column count: {_rp_total} (OK)")

FIELD_CLASSIFICATION replay_players_raw column count: 25 (OK)


## Section 9: Write JSON and Markdown Artifacts

In [54]:
# Build JSON artifact
findings = {
    "step": "01_02_04",
    "dataset": "sc2egset",
    "db_memory_footprint_bytes": db_size_bytes,
    "null_census": {
        "replay_players_raw": {
            "total_rows": total_rows,
            "columns": null_census_df.to_dict(orient="records"),
        },
        "replays_meta_raw_filename": meta_fn_null.to_dict(orient="records")[0],
        "struct_flat": {
            "total_rows": sf_total_rows,
            "columns": sf_null_census_df.to_dict(orient="records"),
        },
    },
    "result_distribution": result_dist.to_dict(orient="records"),
    "non_2p_results": non_2p_results.to_dict(orient="records"),
    "categorical_profiles": {},
    "numeric_stats": {},
    "temporal_range": temporal_range.to_dict(orient="records")[0],
    "monthly_counts": monthly.to_dict(orient="records"),
    "error_census": error_census.to_dict(orient="records")[0],
    "cardinality": all_card_df.to_dict(orient="records"),
    "flagged_columns": flagged.to_dict(orient="records") if len(flagged) > 0 else [],
    "struct_flat_shape": list(struct_flat.shape),
    "game_speed_assertion": "PASSED -- all Faster",
    "duration_stats": {},
    "map_dimensions": map_dims.to_dict(orient="records"),
    "plots": [],
}
sql_queries: dict = {}

In [55]:
# Add categorical profiles to findings
for col, df in rp_cat_results.items():
    findings["categorical_profiles"][col] = df.to_dict(orient="records")
for col, df in meta_cat_results.items():
    findings["categorical_profiles"][col] = df.to_dict(orient="records")

# Add numeric stats
for col, df in rp_num_stats.items():
    findings["numeric_stats"][col] = df.to_dict(orient="records")[0]

# Add duration stats
findings["duration_stats"] = duration_stats.to_dict(orient="records")[0]

# Add zero counts
findings["zero_counts"] = {
    "replay_players_raw": zero_counts.to_dict(orient="records")[0],
    "replays_meta_raw": duration_zero_counts.to_dict(orient="records")[0],
}

# Add duplicate check
findings["duplicate_check"] = duplicate_check.to_dict(orient="records")[0]

# Add field classification
findings["field_classification"] = FIELD_CLASSIFICATION

# Add Undecided/Tie context
findings["undecided_tie_context"] = undecided_tie_context.to_dict(orient="records")

# Add new pass-2 analytics
findings["skew_kurtosis"] = skew_kurt_results
findings["numeric_stats_SQ_no_sentinel"] = sq_no_sentinel.to_dict(orient="records")[0]
findings["mmr_zero_interpretation"] = {
    "by_result": mmr_zero_by_result.to_dict(orient="records"),
    "by_highestLeague": mmr_zero_by_league.to_dict(orient="records"),
}
findings["struct_flat_completeness_note"] = sf_completeness_note
findings["isInClan_distribution"] = is_in_clan_df.to_dict(orient="records")
findings["clanTag_top20"] = clan_tag_top20_df.to_dict(orient="records")

In [56]:
# Collect plot filenames
plot_files = sorted(plots_dir.glob("01_02_04_*.png"))
findings["plots"] = [f.name for f in plot_files]
print(f"Plot files: {findings['plots']}")

Plot files: ['01_02_04_apm_histogram.png', '01_02_04_duration_histogram.png', '01_02_04_highestleague_bar.png', '01_02_04_mmr_histogram.png', '01_02_04_race_bar.png', '01_02_04_region_bar.png', '01_02_04_replays_over_time.png', '01_02_04_result_bar.png', '01_02_04_selectedrace_bar.png', '01_02_04_sq_histogram.png', '01_02_04_supplycapped_histogram.png']


## Systematic Column Profile (Top-5 / Bottom-5)

EDA Manual Section 3.1: top-5 / bottom-5 for every column.
Performance note: game_events_raw (608M rows) may take several minutes
per column for GROUP BY on high-cardinality columns (loop, evtTypeName).
event_data is excluded from top_n/bottom_n (JSON strings — GROUP BY
on 608M opaque payloads is semantically useless and OOM-risk).
Elapsed time printed per column; gate: elapsed < 600s per column.
Invariant #6: all SQL captured in sql_queries for artifact emission.

In [57]:
for _tbl in [
    "replay_players_raw",
    "replays_meta_raw",
    "game_events_raw",
    "tracker_events_raw",
    "message_events_raw",
    "map_aliases_raw",
]:
    _skip = {"event_data"} if _tbl == "game_events_raw" else None
    _specs = [
        {"name": r["column_name"], "dtype": r["column_type"]}
        for _, r in con.execute(f"DESCRIBE {_tbl}").df().iterrows()
    ]
    _census = profile_table(con, _tbl, _specs, skip_topn_columns=_skip)
    findings[f"{_tbl}_census"] = _census["profiles"]
    for _col, _sqls in _census["sql_registry"].items():
        sql_queries[f"census.{_tbl}.{_col}.null"] = _sqls["sql_null"]
        sql_queries[f"census.{_tbl}.{_col}.top_n"] = _sqls["sql_top_n"]
        sql_queries[f"census.{_tbl}.{_col}.bottom_n"] = _sqls["sql_bottom_n"]
    print(f"census complete: {_tbl} ({len(_census['profiles'])} columns)")

22:25:14 INFO rts_predict.common.eda_census: Profiling replay_players_raw.filename (VARCHAR)...


22:25:14 INFO rts_predict.common.eda_census: Profiling replay_players_raw.toon_id (VARCHAR)...


22:25:14 INFO rts_predict.common.eda_census: Profiling replay_players_raw.nickname (VARCHAR)...


22:25:14 INFO rts_predict.common.eda_census: Profiling replay_players_raw.playerID (INTEGER)...


22:25:14 INFO rts_predict.common.eda_census: Profiling replay_players_raw.userID (BIGINT)...


...

22:27:43 INFO rts_predict.common.eda_census: Profiling map_aliases_raw.tournament (VARCHAR)...


22:27:43 INFO rts_predict.common.eda_census: Profiling map_aliases_raw.foreign_name (VARCHAR)...


22:27:43 INFO rts_predict.common.eda_census: Profiling map_aliases_raw.english_name (VARCHAR)...


22:27:43 INFO rts_predict.common.eda_census: Profiling map_aliases_raw.filename (VARCHAR)...


  tracker_events_raw.event_data (VARCHAR): nulls=0, card=54277355, elapsed=28.11s
census complete: tracker_events_raw (4 columns)
  message_events_raw.filename (VARCHAR): nulls=0, card=22260, elapsed=0.05s
  message_events_raw.loop (BIGINT): nulls=0, card=18057, elapsed=0.03s
  message_events_raw.evtTypeName (VARCHAR): nulls=0, card=3, elapsed=0.02s
  message_events_raw.event_data (VARCHAR): nulls=0, card=45856, elapsed=0.03s
census complete: message_events_raw (4 columns)
  map_aliases_raw.tournament (VARCHAR): nulls=0, card=70, elapsed=0.0s
  map_aliases_raw.foreign_name (VARCHAR): nulls=0, card=1488, elapsed=0.0s
  map_aliases_raw.english_name (VARCHAR): nulls=0, card=215, elapsed=0.0s
  map_aliases_raw.filename (VARCHAR): nulls=0, card=70, elapsed=0.0s
census complete: map_aliases_raw (4 columns)


In [58]:
# Write JSON
findings["sql_queries"] = sql_queries
json_path = artifacts_dir / "01_02_04_univariate_census.json"
json_path.write_text(json.dumps(findings, indent=2, default=str))
print(f"JSON artifact written: {json_path}")

JSON artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda/01_02_04_univariate_census.json


In [59]:
# Build markdown artifact with all SQL verbatim (Invariant #6)
md_lines = [
    "# Step 01_02_04 -- Metadata STRUCT Extraction & Replay-Level EDA",
    "",
    "**Dataset:** sc2egset",
    f"**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
    "",
    "---",
    "",
    "## Section A: STRUCT Field Extraction",
    "",
    "```sql",
    STRUCT_FLAT_SQL.strip(),
    "```",
    "",
    f"Result shape: {struct_flat.shape}",
    "",
    "## Database Memory Footprint",
    "",
    f"DuckDB file size: {db_size_bytes:,} bytes ({db_size_mb} MB)",
    "",
]

In [60]:
md_lines += [
    "## Section B: Full NULL Census",
    "",
    "### replay_players_raw",
    "",
    "```sql",
    NULL_CENSUS_SQL.strip(),
    "```",
    "",
    null_census_df.to_markdown(index=False),
    "",
    "### replays_meta_raw.filename",
    "",
    "```sql",
    META_FILENAME_NULL_SQL.strip(),
    "```",
    "",
    meta_fn_null.to_markdown(index=False),
    "",
    "### struct_flat NULL census",
    "",
    "```sql",
    STRUCT_FLAT_NULL_CENSUS_SQL.strip(),
    "```",
    "",
    sf_null_census_df.to_markdown(index=False),
    "",
    "### struct_flat completeness note",
    "",
    (
        "All 17 struct_flat columns have 0 NULLs. No missingness co-occurrence to analyze."
        if _sf_all_zero_nulls
        else "struct_flat has columns with NULLs -- see co-occurrence matrix in JSON artifact."
    ),
    "",
]

In [61]:
md_lines += [
    "## Section C: Target Variable",
    "",
    "```sql",
    RESULT_DIST_SQL.strip(),
    "```",
    "",
    result_dist.to_markdown(index=False),
    "",
    "### Non-2-player replay results",
    "",
    "```sql",
    NON_2P_RESULT_SQL.strip(),
    "```",
    "",
    non_2p_results.to_markdown(index=False),
    "",
    "### Undecided/Tie replay context (corrective query)",
    "",
    "```sql",
    UNDECIDED_TIE_CONTEXT_SQL.strip(),
    "```",
    "",
    undecided_tie_context.to_markdown(index=False),
    "",
    (
        f"Finding: All {len(undecided_tie_context)} rows "
        f"({'Undecided' if (undecided_tie_context['result'] == 'Undecided').all() else 'Undecided and Tie'}) "
        f"came from replays with player_row_count="
        f"{undecided_tie_context['player_row_count'].unique().tolist()} "
        f"and max_players={undecided_tie_context['max_players'].unique().tolist()}."
    ),
    "",
    (
        "Note: The OR-condition non-2-player query above returned only Win/Loss. "
        "Undecided/Tie rows are present in standard 2-player replays per this corrective query."
    ),
    "",
]

In [62]:
md_lines += [
    "## Section D: Categorical Field Profiles",
    "",
]
for col in RP_CAT_COLS:
    cat_sql = f"""\
SELECT {col}, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM replay_players_raw
GROUP BY {col}
ORDER BY cnt DESC"""
    md_lines += [
        f"### {col}",
        "",
        "```sql",
        cat_sql,
        "```",
        "",
        rp_cat_results[col].to_markdown(index=False),
        "",
    ]

In [63]:
md_lines += [
    "### isInClan distribution",
    "",
    "```sql",
    IS_IN_CLAN_SQL.strip(),
    "```",
    "",
    is_in_clan_df.to_markdown(index=False),
    "",
    "### clanTag top-20",
    "",
    "```sql",
    CLAN_TAG_TOP20_SQL.strip(),
    "```",
    "",
    clan_tag_top20_df.to_markdown(index=False),
    "",
]

In [64]:
for col in META_CAT_COLS:
    if col == "map_name":
        cat_sql = f"""\
SELECT {col}, COUNT(*) AS cnt
FROM struct_flat
GROUP BY {col}
ORDER BY cnt DESC
LIMIT 20"""
    else:
        cat_sql = f"""\
SELECT {col}, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM struct_flat
GROUP BY {col}
ORDER BY cnt DESC"""
    md_lines += [
        f"### {col}",
        "",
        "```sql",
        cat_sql,
        "```",
        "",
        meta_cat_results[col].to_markdown(index=False),
        "",
    ]

In [65]:
md_lines += [
    "### Game Speed Assertion",
    "",
    "```python",
    'assert set(speed_counts["game_speed"].dropna()) == {"Faster"}',
    "```",
    "",
    "**PASSED** -- all replays confirmed Faster speed.",
    "",
]

In [66]:
md_lines += [
    "## Section E: Numeric Descriptive Statistics",
    "",
    (
        "Note: APM, SQ, and supplyCappedPercent are in-game-only fields. "
        "See `field_classification` in the JSON artifact for the full "
        "pre-game/in-game/post-game/identifier/target/constant taxonomy."
    ),
    "",
]
for col in RP_NUM_COLS:
    num_sql = f"""\
SELECT
    MIN({col}) AS min_val, MAX({col}) AS max_val,
    ROUND(AVG({col}), 2) AS mean_val,
    ROUND(MEDIAN({col}), 2) AS median_val,
    ROUND(STDDEV({col}), 2) AS stddev_val,
    PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY {col}) AS p05,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY {col}) AS p25,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY {col}) AS p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {col}) AS p95
FROM replay_players_raw
WHERE {col} IS NOT NULL"""
    md_lines += [
        f"### {col}",
        "",
        "```sql",
        num_sql,
        "```",
        "",
        rp_num_stats[col].to_markdown(index=False),
        "",
    ]

In [67]:
md_lines += [
    "### elapsed_game_loops (duration)",
    "",
    "```sql",
    DURATION_STATS_SQL.strip(),
    "```",
    "",
    duration_stats.to_markdown(index=False),
    "",
    "### Map Dimensions",
    "",
    "```sql",
    MAP_DIM_SQL.strip(),
    "```",
    "",
    map_dims.to_markdown(index=False),
    "",
    "### SQ stats excluding INT32_MIN sentinel",
    "",
    "```sql",
    SQ_NO_SENTINEL_SQL.strip(),
    "```",
    "",
    sq_no_sentinel.to_markdown(index=False),
    "",
    (
        "The main SQ descriptive statistics (Section E) are contaminated by 2 rows "
        "containing the INT32_MIN sentinel value (-2147483648). The stats above "
        "exclude those rows. The sentinel causes the main SQ mean and stddev to be "
        "misleading. Refer to the sentinel-excluded stats above for the clean SQ "
        "distribution's actual median and range."
    ),
    "",
]

In [68]:
md_lines += [
    "## Section E1: Skewness and Kurtosis (EDA Manual 3.1)",
    "",
    "SQL template (one query per column):",
    "",
    "```sql",
    "SELECT",
    "    ROUND(SKEWNESS({col}), 4) AS skewness,",
    "    ROUND(KURTOSIS({col}), 4) AS kurtosis",
    "FROM {table}",
    "WHERE {col} IS NOT NULL",
    "```",
    "",
    skew_kurt_df.to_markdown(index=False),
    "",
]

In [69]:
md_lines += [
    "## Section E2: Zero Counts (EDA Manual 3.1)",
    "",
    "### replay_players_raw zero counts",
    "",
    "```sql",
    ZERO_COUNT_SQL.strip(),
    "```",
    "",
    zero_counts.to_markdown(index=False),
    "",
    "### elapsed_game_loops zero count",
    "",
    "```sql",
    DURATION_ZERO_COUNT_SQL.strip(),
    "```",
    "",
    duration_zero_counts.to_markdown(index=False),
    "",
    "### MMR zero-spike interpretation",
    "",
    "```sql",
    MMR_ZERO_BY_RESULT_SQL.strip(),
    "```",
    "",
    mmr_zero_by_result.to_markdown(index=False),
    "",
    "```sql",
    MMR_ZERO_BY_LEAGUE_SQL.strip(),
    "```",
    "",
    mmr_zero_by_league.to_markdown(index=False),
    "",
    (
        "Interpretation: If MMR=0 rate is uniform across all result categories (~83%), "
        "that supports the 'sentinel = not reported' hypothesis rather than a "
        "loss-correlated sentinel."
    ),
    "",
    "## Section E3: Duplicate Detection",
    "",
    "```sql",
    DUPLICATE_CHECK_SQL.strip(),
    "```",
    "",
    duplicate_check.to_markdown(index=False),
    "",
]

In [70]:
md_lines += [
    "## Section F: Temporal Range",
    "",
    "```sql",
    TEMPORAL_RANGE_SQL.strip(),
    "```",
    "",
    temporal_range.to_markdown(index=False),
    "",
    "### Monthly Replay Counts",
    "",
    "```sql",
    MONTHLY_COUNTS_SQL.strip(),
    "```",
    "",
    monthly.to_markdown(index=False),
    "",
]

In [71]:
md_lines += [
    "## Section G: Error Column Census",
    "",
    "```sql",
    ERROR_CENSUS_SQL.strip(),
    "```",
    "",
    error_census.to_markdown(index=False),
    "",
]

In [72]:
md_lines += [
    "## Section H: Dead/Constant/Near-Constant Field Detection",
    "",
    "Cardinality query (per column):",
    "",
    "```sql",
    "SELECT '{col}' AS column_name,",
    "       COUNT(DISTINCT {col}) AS cardinality,",
    "       COUNT(*) AS total_rows,",
    "       ROUND(COUNT(DISTINCT {col})::DOUBLE / COUNT(*)::DOUBLE, 6)"
    " AS uniqueness_ratio",
    "FROM {table}",
    "```",
    "",
    "### Full Cardinality Table",
    "",
    all_card_df.to_markdown(index=False),
    "",
    "### Flagged Columns",
    "",
    "### Interpretation Note",
    "",
    (
        "The uniqueness_ratio < 0.001 threshold (EDA Manual Section 3.3) flags all "
        "low-cardinality columns mechanically. This includes:"
    ),
    "",
    (
        "- **Expected categoricals** (race, selectedRace, highestLeague, region, realm, "
        "result, isInClan, color_*, startDir): These are inherently low-cardinality "
        "fields in any game dataset. Flagging them is a consequence of the threshold "
        "definition, not a data quality concern. Their value distributions are profiled "
        "in Section D."
    ),
    "",
    (
        "- **Genuinely constant/degenerate** (game_speed, game_speed_init, "
        "gameEventsErr, messageEventsErr, trackerEvtsErr): cardinality=1 fields "
        "that carry zero information and should be excluded from feature engineering."
    ),
    "",
    (
        "- **Low-cardinality numerics** (playerID, userID, handicap): "
        "playerID ranges 1-16 (slot index within replay, not a player identifier); "
        "userID is similarly replay-scoped; handicap is 100 for all but 1 row. "
        "These warrant investigation in cleaning (01_04) but are not data quality "
        "defects per se."
    ),
    "",
    (
        "Downstream steps should use the `field_classification` in the JSON artifact "
        "rather than the near-constant flag to decide feature eligibility."
    ),
    "",
    (
        "**Threshold sensitivity note:** The uniqueness_ratio < 0.001 threshold "
        "is appropriate for this dataset (N=44,817 rows). For larger datasets "
        "(N > 1M), the same threshold would flag more columns as near-constant "
        "because uniqueness_ratio = cardinality / N decreases with N even for "
        "columns with stable cardinality. Re-evaluate the threshold for each "
        "dataset based on its row count and the cardinality distribution."
    ),
    "",
]
if len(flagged) > 0:
    md_lines.append(flagged.to_markdown(index=False))
else:
    md_lines.append("(none)")
md_lines.append("")

In [73]:
# Write markdown
md_path = artifacts_dir / "01_02_04_univariate_census.md"
md_path.write_text("\n".join(md_lines))
print(f"Markdown artifact written: {md_path}")

Markdown artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda/01_02_04_univariate_census.md


In [74]:
con.close()
print("Done. All artifacts written.")

Done. All artifacts written.
